<a href="https://colab.research.google.com/github/awfajri/artificial-intelligence/blob/main/Studi_kasus_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Studi kasus 2 - Auf Fajri Ramadhani




Kelas : 4F

NPM : 2410631170059



---



#**soal**
Seorang penjual baju online ingin memprediksi apakah seorang pembeli akan membeli
produknya atau tidak. Penjual tersebut memiliki dataset pembeli yang telah membeli dan tidak
membeli produk. Dataset tersebut terdiri dari variabel independen berupa usia dan penghasilan,
serta variabel dependen berupa pembelian (1=ya, 0=tidak). Gunakan metode Naive Bayes
untuk memprediksi apakah seseorang akan membeli produk dari toko online tersebut
berdasarkan usia dan penghasilan.

**Jawab**

1. prepare data & library

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt, exp, pi

df = pd.read_csv('employee.csv')

df['Beli'] = [1 if x > 4000 else 0 for x in df['Salary']]
dataset = df[['Age', 'Salary', 'Beli']].values.tolist()

Tahap awal adalah mengimpor modul math untuk perhitungan rumus Gaussian dan pandas untuk membaca file employee.csv . Di sini saya juga menyiapkan data dengan mengambil kolom Age sebagai Usia dan Salary sebagai Penghasilan, serta membuat label target "Beli" secara otomatis agar model bisa belajar.

2. Peringkasan Data Berdasarkan Karakteristik Pembeli

In [ ]:
def summarize_by_class(dataset):
    separated = {}
    for i in range(len(dataset)):
        vector = dataset[i]
        class_value = vector[-1]
        if class_value not in separated:
            separated[class_value] = list()
        separated[class_value].append(vector)

    summaries = {}
    for class_value, rows in separated.items():
        summaries[class_value] = [(np.mean(col), np.std(col), len(col)) for col in zip(*rows)]
        del(summaries[class_value][-1])
    return summaries

Fungsi ini akan membagi data menjadi dua kelompok: kelompok yang membeli (1) dan tidak membeli (0). Untuk masing-masing kelompok, sistem akan menghitung rata-rata (mean) dan standar deviasi dari usia dan penghasilan mereka . Hal ini penting agar AI tahu "ciri-ciri" orang yang biasanya membeli baju di toko tersebut.

3. Perhitungan Peluang dengan Rumus Gaussian

In [ ]:
def calculate_probability(x, mean, stdev):
    if stdev == 0: stdev = 0.0001 # Menghindari pembagian dengan nol
    exponent = exp(-((x-mean)**2 / (2 * stdev**2)))
    return (1 / (sqrt(2 * pi) * stdev)) * exponent

Karena usia dan penghasilan adalah angka desimal/kontinu, saya menggunakan fungsi kepadatan probabilitas Gaussian . Fungsi ini akan menghitung seberapa besar peluang data calon pembeli baru (misal: usia 40) cocok dengan data rata-rata pembeli yang sudah ada di database .

4. Proses Prediksi Keputusan Akhir


In [ ]:
def predict(summaries, row):
    total_rows = sum([summaries[label][0][2] for label in summaries])
    probabilities = {}
    for class_value, class_summaries in summaries.items():
        probabilities[class_value] = summaries[class_value][0][2]/float(total_rows)
        for i in range(len(class_summaries)):
            mean, stdev, _ = class_summaries[i]
            probabilities[class_value] *= calculate_probability(row[i], mean, stdev)

    return max(probabilities, key=probabilities.get)

Fungsi ini akan menghitung total probabilitas untuk kedua opsi (Beli atau Tidak). Sistem akan mengalikan peluang usia dan peluang penghasilan terhadap masing-masing kelas . Akhirnya, sistem akan memilih keputusan yang memiliki nilai probabilitas paling tinggi sebagai hasil prediksinya .

5. Eksekusi Program

In [ ]:
# Training Model
model = summarize_by_class(dataset)

# Data Calon Pembeli Baru: [Usia, Penghasilan]
new_buyer = [40, 5500]

# Melakukan Prediksi
hasil = predict(model, new_buyer)
print(f"Data Calon Pembeli: Usia={new_buyer[0]}, Penghasilan={new_buyer[1]}")
print(f"Hasil Prediksi: {'AKAN MEMBELI (1)' if hasil == 1 else 'TIDAK MEMBELI (0)'}")

Data Calon Pembeli: Usia=40, Penghasilan=5500
Hasil Prediksi: AKAN MEMBELI (1)


Di tahap akhir, kita menjalankan model dengan data dari employee.csv. Kita mencoba memprediksi seorang calon pembeli baru yang memiliki Usia 40 tahun dan Penghasilan 5500